# Notebook 2 – Numerical Feature Engineering

## Learning objectives
Create numerical features using arithmetic, ratios, percentages, aggregations, mathematical transformations, rates, and normalization. For each technique, consider when and why it is appropriate.

## 1. Sample Dataset

### Example
An online store records item price, quantity, discount, shipping cost, and previous-period sales. These values can be combined into business features.

In [1]:
import pandas as pd
import numpy as np

sales = pd.DataFrame({
    "unit_price": [100, 250, 80, 500, 120],
    "quantity": [2, 1, 5, 2, 3],
    "discount": [10, 25, 0, 50, 12],
    "shipping": [20, 30, 15, 40, 18],
    "previous_sales": [180, 300, 350, 900, 330],
    "current_sales": [200, 250, 400, 1000, 360],
    "days": [10, 5, 20, 25, 12]
})
sales

,unit_price,quantity,discount,shipping,previous_sales,current_sales,days
0,100,2,10,20,180,200,10
1,250,1,25,30,300,250,5
2,80,5,0,15,350,400,20
3,500,2,50,40,900,1000,25
4,120,3,12,18,330,360,12


### Output explanation
Each row represents a sales record. The columns provide numeric inputs for creating additional features.

## 2. Addition

Addition combines quantities that contribute to a total.

### Example
Total order cost before discount can be calculated as item value plus shipping.

In [2]:
sales["item_value"] = sales["unit_price"] * sales["quantity"]
sales["subtotal_with_shipping"] = sales["item_value"] + sales["shipping"]
sales[["item_value", "shipping", "subtotal_with_shipping"]]

,item_value,shipping,subtotal_with_shipping
0,200,20,220
1,250,30,280
2,400,15,415
3,1000,40,1040
4,360,18,378


### Output explanation
The new total combines item value and shipping. Use this when both amounts belong to the same business total.

## 3. Subtraction and Differences

Subtraction measures the gap between two numeric values.

### Example
Net item value after a fixed discount amount is item value minus discount.

In [3]:
sales["net_item_value"] = sales["item_value"] - sales["discount"]
sales["sales_difference"] = sales["current_sales"] - sales["previous_sales"]
sales[["item_value", "discount", "net_item_value", "sales_difference"]]

,item_value,discount,net_item_value,sales_difference
0,200,10,190,20
1,250,25,225,-50
2,400,0,400,50
3,1000,50,950,100
4,360,12,348,30


### Output explanation
A positive sales difference indicates an increase from the previous period; a negative value indicates a decrease. Confirm that the compared periods have matching definitions.

## 4. Multiplication

Multiplication combines a quantity and a per-unit measure.

### Example
Revenue from a line item is often quantity multiplied by unit price.

In [4]:
sales["line_revenue"] = sales["unit_price"] * sales["quantity"]
sales[["unit_price", "quantity", "line_revenue"]]

,unit_price,quantity,line_revenue
0,100,2,200
1,250,1,250
2,80,5,400
3,500,2,1000
4,120,3,360


### Output explanation
The result estimates the line revenue before any discount or extra charge. Use it when unit price and quantity refer to the same item and transaction.

## 5. Division and Ratios

Division expresses one value relative to another. Always consider zero denominators and the meaning of the units.

### Example
Average price per item can be calculated as total item value divided by quantity.

In [5]:
sales["average_price_per_item"] = (
    sales["item_value"] / sales["quantity"].replace(0, np.nan)
)
sales["shipping_to_item_ratio"] = (
    sales["shipping"] / sales["item_value"].replace(0, np.nan)
)
sales[["average_price_per_item", "shipping_to_item_ratio"]]

,average_price_per_item,shipping_to_item_ratio
0,100.0,0.1000
1,250.0,0.1200
2,80.0,0.0375
3,500.0,0.0400
4,120.0,0.0500


### Output explanation
The average price is a per-item value. The shipping ratio expresses shipping cost relative to item value; `NaN` protects against division by zero.

## 6. Percentages

A percentage expresses a part relative to a total, multiplied by 100.

### Example
Discount percentage is discount divided by item value, multiplied by 100.

In [6]:
sales["discount_percent"] = (
    sales["discount"] / sales["item_value"].replace(0, np.nan)
) * 100
sales[["item_value", "discount", "discount_percent"]]

,item_value,discount,discount_percent
0,200,10,5.000000
1,250,25,10.000000
2,400,0,0.000000
3,1000,50,5.000000
4,360,12,3.333333


### Output explanation
The feature represents the discount as a percentage of item value. Check whether the discount column is an amount or already a percentage before applying this formula.

## 7. Aggregations

Aggregations summarize multiple numeric observations using statistics such as sum, mean, median, minimum, maximum, and standard deviation.

### Example
Calculate overall summary statistics for current sales.

In [7]:
summary = sales["current_sales"].agg(
    ["count", "sum", "mean", "median", "min", "max", "std"]
)
summary

count        5.000000
sum       2210.000000
mean       442.000000
median     360.000000
min        200.000000
max       1000.000000
std        322.211111
Name: current_sales, dtype: float64

### Output explanation
The result summarizes the distribution of current sales. In real projects, group-level aggregates must be calculated using only information available at prediction time.

## 8. Log Transformation

A log transformation can reduce right skew and compress very large positive values. `log1p(x)` calculates `log(1 + x)` and supports zero.

### Example
Transaction amounts often contain a small number of very large values.

In [8]:
amount = pd.Series([0, 10, 25, 100, 1000, 10000], name="amount")
log_data = pd.DataFrame({
    "amount": amount,
    "log1p_amount": np.log1p(amount)
})
log_data

,amount,log1p_amount
0,0,0.000000
1,10,2.397895
2,25,3.258097
3,100,4.615121
4,1000,6.908755
5,10000,9.210440


### Output explanation
Large values are compressed more strongly than small values. Log transformation is not appropriate for arbitrary negative values without a suitable domain-specific adjustment.

## 9. Square Root Transformation

The square root transformation can reduce right skew, especially for non-negative count-like variables.

### Example
Transform the number of support tickets.

In [9]:
tickets = pd.Series([0, 1, 4, 9, 25, 100], name="ticket_count")
pd.DataFrame({
    "ticket_count": tickets,
    "sqrt_ticket_count": np.sqrt(tickets)
})

,ticket_count,sqrt_ticket_count
0,0,0.0
1,1,1.0
2,4,2.0
3,9,3.0
4,25,5.0
5,100,10.0


### Output explanation
The square root reduces the spread between large and small counts. Apply it only when the values are non-negative.

## 10. Power Transformation

Power transformations change a feature's distribution. Scikit-learn provides Yeo-Johnson (supports positive and negative values) and Box-Cox (requires strictly positive values).

### Example
Use Yeo-Johnson to transform a skewed feature.

In [10]:
from sklearn.preprocessing import PowerTransformer

skewed = pd.DataFrame({"value": [ -3, 0, 1, 2, 5, 20, 100]})
power = PowerTransformer(method="yeo-johnson")
skewed["yeo_johnson_value"] = power.fit_transform(skewed[["value"]])
skewed

,value,yeo_johnson_value
0,-3,-1.772467
1,0,-0.426394
2,1,-0.247419
3,2,-0.125664
4,5,0.116881
5,20,0.692643
6,100,1.762420


### Output explanation
The transformer learns a power parameter from the sample and returns a transformed column. In a predictive workflow, fit it on training data only.

## 11. Absolute Difference

Absolute difference measures the size of a gap without considering its direction.

### Example
Compare current and previous sales magnitude.

In [11]:
sales["absolute_sales_difference"] = (
    sales["current_sales"] - sales["previous_sales"]
).abs()
sales[["current_sales", "previous_sales", "absolute_sales_difference"]]

,current_sales,previous_sales,absolute_sales_difference
0,200,180,20
1,250,300,50
2,400,350,50
3,1000,900,100
4,360,330,30


### Output explanation
The result is always zero or positive. It is useful when the size of the change matters more than whether sales increased or decreased.

## 12. Relative Difference

Relative difference expresses a change relative to a baseline. A common percentage-change formula is `(new - old) / old`.

### Example
Calculate percentage sales change from the previous period.

In [12]:
sales["relative_sales_change"] = (
    (sales["current_sales"] - sales["previous_sales"])
    / sales["previous_sales"].replace(0, np.nan)
)
sales["sales_change_percent"] = sales["relative_sales_change"] * 100
sales[["relative_sales_change", "sales_change_percent"]]

,relative_sales_change,sales_change_percent
0,0.111111,11.111111
1,-0.166667,-16.666667
2,0.142857,14.285714
3,0.111111,11.111111
4,0.090909,9.090909


### Output explanation
A value of `0.20` means a 20% increase; `-0.10` means a 10% decrease. A zero baseline produces a missing value rather than division by zero.

## 13. Rate Features

A rate measures an amount per unit of time or another exposure measure.

### Example
Calculate sales per day.

In [13]:
sales["sales_per_day"] = (
    sales["current_sales"] / sales["days"].replace(0, np.nan)
)
sales[["current_sales", "days", "sales_per_day"]]

,current_sales,days,sales_per_day
0,200,10,20.0
1,250,5,50.0
2,400,20,20.0
3,1000,25,40.0
4,360,12,30.0


### Output explanation
This feature compares sales records with different durations on a per-day basis. The denominator must represent a meaningful exposure period.

## 14. Normalized Features

Normalization places values on a common scale. Min-Max scaling maps values to a chosen range, commonly 0 to 1. Standardization centers values around zero with unit variance.

### Example
Scale current sales using MinMaxScaler and StandardScaler.

In [14]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

numeric = sales[["current_sales"]]
minmax = MinMaxScaler()
standard = StandardScaler()

scaled = pd.DataFrame({
    "current_sales": sales["current_sales"],
    "minmax_scaled": minmax.fit_transform(numeric).ravel(),
    "standard_scaled": standard.fit_transform(numeric).ravel()
})
scaled

,current_sales,minmax_scaled,standard_scaled
0,200,0.0000,-0.839711
1,250,0.0625,-0.666217
2,400,0.2500,-0.145735
3,1000,1.0000,1.936193
4,360,0.2000,-0.284530


### Output explanation
Min-Max values fall between 0 and 1 for this fitted sample. Standardized values are centered around zero. Fit scalers on training data only to avoid train-test leakage.

## 15. Feature Justification Checklist

For every engineered feature, document:
- **Feature name:** what is the new column called?
- **Source columns:** which raw columns were used?
- **Logic/formula:** how is it calculated?
- **Reason:** why was it created?
- **Expected relationship with target:** what relationship is plausible?
- **Business meaning:** what does it represent?
- **ML value:** why might it help?
- **Leakage risk:** is it available at prediction time?
- **Decision:** retain, remove, or investigate further.

### Example
` sales_per_day ` uses `current_sales` and `days`. It can help compare records with different durations, but only if both values are known at prediction time.